<a href="https://colab.research.google.com/github/toshniwal-keshav/LabArtificialInteligence/blob/main/Week7_FirstOrderLogic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

class LogicVar:
    def __init__(self, label):
        self.label = label

    def __str__(self):
        return self.label

    def __repr__(self):
        return f"Var({self.label})"

    def __eq__(self, other):
        return isinstance(other, LogicVar) and self.label == other.label

    def __hash__(self):
        return hash(self.label)


class LogicConst:
    def __init__(self, label):
        self.label = label

    def __str__(self):
        return self.label

    def __repr__(self):
        return f"Const({self.label})"

    def __eq__(self, other):
        return isinstance(other, LogicConst) and self.label == other.label

    def __hash__(self):
        return hash(self.label)


class LogicFn:
    def __init__(self, name, parameters):
        self.name = name
        self.parameters = list(parameters)

    def __str__(self):
        return f"{self.name}({', '.join(str(p) for p in self.parameters)})"

    def __repr__(self):
        return f"Fn({self.name}, {self.parameters})"

    def __eq__(self, other):
        return isinstance(other, LogicFn) and self.name == other.name and self.parameters == other.parameters


def check_cycle(variable, term, env):
    """Occurs check: detects if 'variable' is nested inside 'term' under current environment bindings."""
    if variable == term:
        return True
    if isinstance(term, LogicVar) and term in env:
        return check_cycle(variable, env[term], env)
    if isinstance(term, LogicFn):
        return any(check_cycle(variable, child, env) for child in term.parameters)
    return False


def bind_var(variable, expression, env):
    """Binds a LogicVar to an expression if valid, resolving existing bindings and performing occurs checks."""
    if variable in env:
        return solve_unify(env[variable], expression, env)
    if isinstance(expression, LogicVar) and expression in env:
        return solve_unify(variable, env[expression], env)
    if check_cycle(variable, expression, env):
        return None

    updated_env = env.copy()
    updated_env[variable] = expression
    return updated_env


def solve_unify(t1, t2, env=None):
    """Primary logic engine to unify structure t1 and t2 with substitution dictionary 'env'."""
    if env is None:
        env = {}

    if t1 == t2:
        return env
    if isinstance(t1, LogicVar):
        return bind_var(t1, t2, env)
    if isinstance(t2, LogicVar):
        return bind_var(t2, t1, env)

    if isinstance(t1, LogicFn) and isinstance(t2, LogicFn):
        if t1.name != t2.name or len(t1.parameters) != len(t2.parameters):
            return None
        current_env = env
        for p1, p2 in zip(t1.parameters, t2.parameters):
            current_env = solve_unify(p1, p2, current_env)
            if current_env is None:
                return None
        return current_env

    return None


def tokenize_and_parse(token_str):
    """Converts a logical string representation into nested instances of LogicVar, LogicConst, or LogicFn."""
    token_str = token_str.strip().replace(" ", "")
    if not token_str:
        return None

    outer_pattern = re.match(r'^([a-zA-Z0-9_]+)\((.*)\)$', token_str)
    if outer_pattern:
        fn_name = outer_pattern.group(1)
        nested_str = outer_pattern.group(2)

        elements = []
        buffer = []
        depth = 0
        for char in nested_str:
            if char == '(':
                depth += 1
            elif char == ')':
                depth -= 1

            if char == ',' and depth == 0:
                elements.append(tokenize_and_parse("".join(buffer)))
                buffer = []
            else:
                buffer.append(char)
        if buffer:
            elements.append(tokenize_and_parse("".join(buffer)))

        return LogicFn(fn_name, elements)

    if token_str[0].isupper():
        return LogicVar(token_str)
    return LogicConst(token_str)


def start_terminal_app():
    """Interactive console interface taking raw inputs from user and matching them."""
    print("=================== LOGICAL UNIFICATION TERMINAL ===================")
    print("Note:")
    print(" - Variables must start with UPPERCASE characters (e.g. X, Y, Z)")
    print(" - Constants & Functions must start with lowercase characters (e.g. a, f(X))")
    print(" - Type 'exit' to terminate the app.\n")

    while True:
        stmt1 = input("Enter First Statement: ").strip()
        if stmt1.lower() == 'exit' or not stmt1:
            print("Terminating. Goodbye!")
            break

        stmt2 = input("Enter Second Statement: ").strip()
        if stmt2.lower() == 'exit' or not stmt2:
            print("Terminating. Goodbye!")
            break

        try:
            ast1 = tokenize_and_parse(stmt1)
            ast2 = tokenize_and_parse(stmt2)

            print(f"Parsed First: {ast1}")
            print(f"Parsed Second: {ast2}")

            results = solve_unify(ast1, ast2)
            if results is not None:
                clean_bindings = {str(k): str(v) for k, v in results.items()}
                print("RESULT: SUCCESSFUL UNIFICATION!")
                print(f"SUBSTITUTIONS: {clean_bindings}\n")
            else:
                print("RESULT: FAILED TO UNIFY (No compatible substitution exists).\n")
        except Exception as error:
            print(f"Error processing terms: {error}. Please check syntax formatting.\n")

# Launch the program inside Colab terminal input context
start_terminal_app()

=================== LOGICAL UNIFICATION TERMINAL ===================
Note:
 - Variables must start with UPPERCASE characters (e.g. X, Y, Z)
 - Constants & Functions must start with lowercase characters (e.g. a, f(X))
 - Type 'exit' to terminate the app.

Enter First Statement: p(b, X, f(g(Z)))
Enter Second Statement: p( Z, f(Y) , f(Y))
Parsed First: p(b, X, f(g(Z)))
Parsed Second: p(Z, f(Y), f(Y))
RESULT: SUCCESSFUL UNIFICATION!
SUBSTITUTIONS: {'Z': 'b', 'X': 'f(Y)', 'Y': 'g(Z)'}

Enter First Statement: p(  y, G(A,B))
Enter Second Statement: p(G(x, x), y)
Parsed First: p(y, G(A, B))
Parsed Second: p(G(x, x), y)
RESULT: FAILED TO UNIFY (No compatible substitution exists).



KeyboardInterrupt: Interrupted by user